In [1]:
from metasmith.models.solver import solve_by_mcts, Transform, Endpoint

transforms = []

tr = Transform()
tr.AddRequirement(properties={"start"})
tr.AddProduct(properties={"a"})
tr.NewProductGroup()
tr.AddProduct(properties={"b"})
transforms.append(tr)

tr = Transform()
tr.AddRequirement(properties={"a"})
tr.AddProduct(properties={"x"})
transforms.append(tr)

tr = Transform()
tr.AddRequirement(properties={"b"})
tr.AddProduct(properties={"x"})
transforms.append(tr)

given = [Endpoint(properties={"start"})]
target = Transform()
target.AddRequirement(properties={"x"})
res = solve_by_mcts(
    given=given,
    target=target,
    transforms=transforms,
)

for i, plan in enumerate(res.dependency_plans):
    print(f">>> {i+1}")
    for appl in plan:
        print(appl.transform)

>>> 1
->{start}
{start}->{a}|{b}
{a}->{x}
{x}->
>>> 2
->{start}
{start}->{a}|{b}
{b}->{x}
{x}->


In [9]:
from pathlib import Path
from metasmith.python_api import Agent, Source, DataInstanceLibrary, TransformInstanceLibrary, WorkflowTask
from metasmith.python_api import DataTypeLibrary, Endpoint
from metasmith.python_api import Resources, Size, Duration
from local.constants import WORKSPACE_ROOT

path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
# smith.Deploy()

In [10]:

import numpy as np
test = [np.array([1,2,3]), np.array([7,8,9])]

res = np.vstack([s for s in test]).mean(axis=0)
res

array([4., 5., 6.])

In [11]:
x = object()

x == x

True

In [12]:
mock_types = DataTypeLibrary(
    types={
        k:Endpoint({k}|v)
        for k, v in [
            ("start", {"test"}),
            ("a", {"test"}),
            ("b", {"test"}),
            ("x", {"test"}),
            ("target", {"test"}),
        ]
    }
)

transforms = TransformInstanceLibrary("./transforms/branching", include_std=False)
transforms.AddTypeLibrary("mock", mock_types)
transforms.Save() # updates types
transforms.AddStub("branch")
transforms.AddStub("a_no_op")
transforms.AddStub("b_no_op")
transforms.AddStub("gather")
transforms.Save()

In [13]:
inputs = DataInstanceLibrary("./cache/dev21.mock.xgdb")
inputs.AddTypeLibrary("mock", mock_types)
samples = []
# for i in range(6):
for i in range(1):
    in_path = WORKSPACE_ROOT/f"main/local_mock/cache/test/mock d.{i}"
    in_path.parent.mkdir(exist_ok=True)
    with open(in_path, "w") as f:
        f.write("0")
    # t = "mock::a" if i%3==0 else "mock::x"
    # inputs.AddItem(in_path, t)
    # inputs.AddItem(in_path, "mock::a")
    inputs.AddItem(in_path, "mock::start")
    # inputs.AddItem(in_path, "mock::x")
    samples.append(in_path)
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, e, e.parents)

mock::start <[start,test]:XtzDM2Ef> set()


In [14]:
# inputs.Consolidate()

In [15]:
for loc, t,  in transforms.IterateTransforms():
    print(t.name, t.model)

branch {start-test}->{a-test}|{b-test}
a_no_op {a-test}->{test-x}
b_no_op {b-test}->{test-x}
gather {start-test},{test-x}->{target-test}


In [16]:
task = smith.GenerateWorkflow(
    samples    = [inputs.AsView({p}) for p in samples],
    resources  = [],
    transforms = [transforms],
    # targets    = [mock_types["b"]]
    targets    = [mock_types["x"]]
)
print(task.GetKey(), sum(len(p.steps) for g in task.plans for p in g))

QGlIeKY1 0


In [ ]:
# smith.StageWorkflow(task, on_exist='update_workflow', verify_external_paths=False)
smith.StageWorkflow(task, on_exist='clear', verify_external_paths=False)

In [ ]:
smith.RunWorkflow(
    # task="mVO8fJEE",
    task=task,
    config_file=smith.GetNxfConfigPresets()["local"],
    resource_overrides={
        # transforms
        "all": Resources(
            cpus=1,
            memory=Size.GB(1),
        ),

        # # all of same transform
        # transforms["no_op_no_fail"]: Resources(
        #     memory=Size.GB(3),
        # ),

        # # all of same transform in batch
        # transforms["no_op"]: {
        #     1: Resources(
        #         memory=Size.GB(2),
        #     ),
        # },

        # # specifc step
        # (1, 1): Resources(
        #     memory=Size.GB(2),
        # ),
    },
)

In [ ]:
with open(WORKSPACE_ROOT/"secrets/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()

params = dict(
    slurmAccount = SLURM_ACCOUNT,
    executor_queueSize = 100,
    process = dict(
        tries=3,
        array=5,
        cpus=1,
        memory='32 GB',
        time='12hours',
    ),
)
smith.RunWorkflow(task, smith.GetNxfConfigPresets()["slurm"], params)